# 02 · MVTec AD - Preprocessing & DataLoaders

For anomaly detection we train ONLY on good images.
The test set is mixed (good + defects), evaluated by AUROC.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
device = get_device()
ASSETS = "cv/mvtec"


In [ ]:
DATA_DIR    = Path('../../../data/05_mvtec/raw')
CATEGORY    = 'transistor'  # One category for the pilot
CAT_DIR     = DATA_DIR / CATEGORY

IMG_SIZE    = 224
BATCH_SIZE  = 16
SEED        = 42
torch.manual_seed(SEED)
print(f'Working with category: {CATEGORY}')

## 1. Custom Dataset for MVTec

In [ ]:
class MVTecDataset(Dataset):
    """MVTec AD dataset.
    Args:
        root: category (e.g. DATA_DIR/'transistor')
        split: 'train' | 'test'
        transform: torchvision transforms
        only_good: if True, good images only (for train)
    """
    def __init__(self, root: Path, split: str = 'train',
                 transform=None, only_good: bool = True):
        self.root      = root
        self.transform = transform
        self.samples   = []  # list of (img_path, label, defect_type)

        split_dir = root / split
        for defect_dir in sorted(split_dir.iterdir()):
            if not defect_dir.is_dir():
                continue
            is_good = defect_dir.name == 'good'
            if only_good and not is_good:
                continue
            label = 0 if is_good else 1  # 0=normal, 1=anomaly
            for img_path in sorted(defect_dir.glob('*.png')):
                self.samples.append((img_path, label, defect_dir.name))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, defect = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, defect

print('MVTecDataset defined.')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
train_set = MVTecDataset(CAT_DIR, split='train', transform=train_transforms, only_good=True)
test_set  = MVTecDataset(CAT_DIR, split='test',  transform=test_transforms,  only_good=False)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train (good only): {len(train_set)}')
print(f'Test (all types):  {len(test_set)}')

# Check one batch
images, labels, defects = next(iter(train_loader))
print(f'Batch shape: {images.shape}')  # [16, 3, 224, 224]

## Summary

| Parameter | Value |
|---|---|
| Train | Good (normal) only |
| Test | Good (0) + Defects (1) - binary label |
| Evaluation | AUROC (ROC AUC) - the standard for anomaly detection |

➡️ **Next step:** `03_mvtec_modeling.ipynb`